# F1 Strategy: GRU Multi-Task Model + Strategy Simulator

This notebook combines:
- GRU + MLP multi-task model (stops cls, tire cls, time reg)
- Strategy enumeration/simulation (0..3 stops)
- Ranked output with total predicted race time and stint breakdown
- Train/validation plots for loss and accuracy

**Extended dataset:** OpenF1 + FastF1 (same schema, concatenated).

In [1]:
import re
from pathlib import Path
from itertools import product
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
 
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

import fastf1
from fastf1 import plotting as ff1_plotting
ff1_plotting.setup_mpl(misc_mpl_mods=False)


/Users/m/miniforge3/envs/tensorflow/lib/python3.10/site-packages/fastf1/plotting/_plotting.py:56: FutureWarning: The `misc_mpl_mods` argument was dropped from `.setup_mpl()` in version 3.6.0 and has no effect anymore. It will be removed in a future version of FastF1.
  warnings.warn(


In [2]:
DATA_DIR = Path("openf1_data")
assert DATA_DIR.exists(), f"Missing: {DATA_DIR.resolve()}"
 
MODEL_FILE = "multitask_strategy_model_improved.keras"
PREPROC_FILE = "multitask_preprocessing_improved.joblib"
 
RACE_CONTEXT_FILE = "race_context_input.csv"
FORECAST_FILE = "forecast_weather_detailed_rainy.csv"
 
WEATHER_FEATURES = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
DRY_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]
DEFAULT_PIT_LOSS = 22.0

# FastF1 cache
FASTF1_CACHE = Path("fastf1_cache")
FASTF1_CACHE.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(FASTF1_CACHE))


In [3]:
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None
 
def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()
 
def safe_mode(series, default="MEDIUM"):
    s = pd.Series(series).dropna()
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else default
 
def format_time(sec):
    m = int(sec // 60)
    s = sec - m * 60
    return f"{m}:{s:06.3f}"


In [4]:
def load_session_maps():
    sessions = safe_read_csv(DATA_DIR / "sessions.csv")
    if sessions.empty:
        return {}, {}, {}
 
    for c in ["session_key", "meeting_key", "session_name", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan
 
    sessions["track_name"] = sessions["circuit_short_name"].fillna(sessions["location"]).fillna(sessions["meeting_key"].astype(str))
    return (
        dict(zip(sessions["session_key"], sessions["track_name"])),
        dict(zip(sessions["session_key"], sessions["meeting_key"])),
        dict(zip(sessions["session_key"], sessions["session_name"].astype(str))),
    )
 
session_to_track, session_to_meeting, session_to_name = load_session_maps()


## Build training data for GRU multi-task (OpenF1 + FastF1)

In [5]:
def get_weather_seq_and_summary(session_key):
    w = safe_read_csv(DATA_DIR / f"weather_session_{session_key}.csv")
    if w.empty:
        return None, None
 
    for c in WEATHER_FEATURES:
        if c not in w.columns:
            w[c] = np.nan
        w[c] = pd.to_numeric(w[c], errors="coerce")
 
    w = w.dropna(subset=WEATHER_FEATURES, how="all")
    if w.empty:
        return None, None
 
    seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
    summary = {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "humidity_mean": float(w["humidity"].mean()),
        "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
    }
    return seq, summary
 
def estimate_time_loss_target(session_key, team_name, driver_team):
    p = safe_read_csv(DATA_DIR / f"pit_session_{session_key}.csv")
    if p.empty or "driver_number" not in p.columns:
        return 0.0
    p = p.merge(driver_team, on="driver_number", how="left")
    p = p[p["team_name"] == team_name]
    if p.empty:
        return 0.0
    for c in ["pit_duration", "duration", "pit_time"]:
        if c in p.columns:
            d = pd.to_numeric(p[c], errors="coerce").dropna()
            if len(d):
                return float(d.sum() + 15.0 * len(d))
    return float(21.0 * len(p))

def build_samples_openf1(race_only=True):
    rows, seqs = [], []
    for sf in sorted(DATA_DIR.glob("stints_session_*.csv")):
        sid = parse_session_key(sf)
        if sid is None:
            continue
        sname = session_to_name.get(sid, "")
        if race_only and ("Race" not in sname and "RACE" not in sname):
            continue
 
        st = safe_read_csv(sf)
        dr = safe_read_csv(DATA_DIR / f"drivers_session_{sid}.csv")
        if st.empty or dr.empty:
            continue
 
        for c in ["driver_number", "compound", "stint_number"]:
            if c not in st.columns:
                st[c] = np.nan
        for c in ["driver_number", "team_name"]:
            if c not in dr.columns:
                dr[c] = np.nan
 
        driver_team = dr[["driver_number", "team_name"]].dropna().drop_duplicates()
        st = st.merge(driver_team, on="driver_number", how="left")
 
        pits = safe_read_csv(DATA_DIR / f"pit_session_{sid}.csv")
        if pits.empty:
            pits = pd.DataFrame(columns=["driver_number"])
        if "driver_number" not in pits.columns:
            pits["driver_number"] = np.nan
        pits = pits.merge(driver_team, on="driver_number", how="left")
 
        grid = safe_read_csv(DATA_DIR / f"starting_grid_session_{sid}.csv")
        pos_col = "position" if "position" in grid.columns else ("grid_position" if "grid_position" in grid.columns else None)
        if not grid.empty and pos_col and "driver_number" in grid.columns:
            g = grid[["driver_number", pos_col]].copy()
            g.columns = ["driver_number", "starting_position"]
            g["starting_position"] = pd.to_numeric(g["starting_position"], errors="coerce")
            g = g.merge(driver_team, on="driver_number", how="left")
            team_pos = g.groupby("team_name")["starting_position"].mean().to_dict()
        else:
            team_pos = {}
 
        seq, ws = get_weather_seq_and_summary(sid)
        if seq is None:
            continue
 
        track = str(session_to_track.get(sid, session_to_meeting.get(sid, "UNKNOWN_TRACK")))
 
        for team, t_st in st.groupby("team_name", dropna=True):
            team = str(team)
            t_pits = pits[pits["team_name"] == team]
            pit_stops = int(len(t_pits))
 
            s1 = t_st[t_st["stint_number"] == 1]
            start_comp = safe_mode(s1["compound"] if len(s1) else t_st["compound"], default="MEDIUM")
            start_comp = str(start_comp).upper()
            if start_comp not in VALID_COMPOUNDS:
                start_comp = "MEDIUM"
 
            non_start = t_st[t_st["compound"].astype(str).str.upper() != start_comp]["compound"]
            target_comp = str(safe_mode(non_start, default=safe_mode(t_st["compound"], "MEDIUM"))).upper()
            if target_comp not in VALID_COMPOUNDS:
                target_comp = "MEDIUM"
 
            rows.append({
                "session_key": sid,
                "team_name": team,
                "track_name": track,
                "starting_position": float(team_pos.get(team, 10.0)),
                "starting_compound": start_comp,
                "pit_stops": pit_stops,
                "target_compound": target_comp,
                "time_loss_target": estimate_time_loss_target(sid, team, driver_team),
                **ws,
            })
            seqs.append(seq)
    return pd.DataFrame(rows), seqs

def build_samples_fastf1(years=range(2018, 2026)):
    rows, seqs = [], []
    for year in years:
        try:
            schedule = fastf1.get_event_schedule(year, include_testing=False)
        except Exception:
            continue

        for _, ev in schedule.iterrows():
            try:
                session = fastf1.get_session(year, ev["EventName"], "R")
                session.load(weather=True, laps=True, telemetry=False)
            except Exception:
                continue

            try:
                weather = session.weather_data
                if weather is None or weather.empty:
                    continue
            except Exception:
                continue

            # Weather sequence and summary
            w = weather.copy()
            w = w.rename(columns={
                "AirTemp": "air_temperature",
                "TrackTemp": "track_temperature",
                "Humidity": "humidity",
                "Rainfall": "rainfall",
                "WindSpeed": "wind_speed",
            })
            for c in WEATHER_FEATURES:
                if c not in w.columns:
                    w[c] = np.nan
                w[c] = pd.to_numeric(w[c], errors="coerce")
            w = w.dropna(subset=WEATHER_FEATURES, how="all")
            if w.empty:
                continue
            seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
            ws = {
                "air_temp_mean": float(w["air_temperature"].mean()),
                "track_temp_mean": float(w["track_temperature"].mean()),
                "humidity_mean": float(w["humidity"].mean()),
                "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean()),
                "wind_speed_mean": float(w["wind_speed"].mean()),
            }

            try:
                laps = session.laps
                if laps is None or laps.empty:
                    continue
            except Exception:
                continue

            # Grid positions from results
            try:
                results = session.results
                if results is None or results.empty:
                    continue
                grid_positions = results.set_index("TeamName")["GridPosition"].to_dict()
            except Exception:
                grid_positions = {}

            track = str(ev.get("EventName", f"{year}_UNKNOWN"))

            # Group by team
            for team, t_laps in laps.groupby("Team", dropna=True):
                team = str(team)

                # Pit stops
                pit_stops = int(t_laps["PitInTime"].notna().sum())

                # Starting compound from first stint
                stint1 = t_laps[t_laps["Stint"] == 1]
                start_comp = safe_mode(stint1["Compound"] if len(stint1) else t_laps["Compound"], default="MEDIUM")
                start_comp = str(start_comp).upper()
                if start_comp not in VALID_COMPOUNDS:
                    start_comp = "MEDIUM"

                # Target compound from non-start stints
                non_start = t_laps[t_laps["Compound"].astype(str).str.upper() != start_comp]["Compound"]
                target_comp = str(safe_mode(non_start, default=safe_mode(t_laps["Compound"], "MEDIUM"))).upper()
                if target_comp not in VALID_COMPOUNDS:
                    target_comp = "MEDIUM"

                # Time loss target (approx: pit count * default loss)
                time_loss_target = float(pit_stops * DEFAULT_PIT_LOSS)

                rows.append({
                    "session_key": f"fastf1_{year}_{ev['EventName']}_{team}",
                    "team_name": team,
                    "track_name": track,
                    "starting_position": float(grid_positions.get(team, 10.0)),
                    "starting_compound": start_comp,
                    "pit_stops": pit_stops,
                    "target_compound": target_comp,
                    "time_loss_target": time_loss_target,
                    **ws,
                })
                seqs.append(seq)

    return pd.DataFrame(rows), seqs

print("Building OpenF1 samples...")
df_openf1, X_seq_openf1 = build_samples_openf1(race_only=True)
print(f"OpenF1 samples: {len(df_openf1)}")

print("Building FastF1 samples...")
df_fastf1, X_seq_fastf1 = build_samples_fastf1(years=range(2018, 2026))
print(f"FastF1 samples: {len(df_fastf1)}")

# Combine datasets
df = pd.concat([df_openf1, df_fastf1], ignore_index=True)
X_seq_raw = X_seq_openf1 + X_seq_fastf1

print(f"Total samples: {len(df)}")
print(df.head())


Building OpenF1 samples...
OpenF1 samples: 90
Building FastF1 samples...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

FastF1 samples: 440
Total samples: 530
  session_key     team_name track_name  starting_position starting_compound  \
0       10033        Alpine      Miami               10.0              HARD   
1       10033  Aston Martin      Miami               10.0              HARD   
2       10033       Ferrari      Miami               10.0              HARD   
3       10033  Haas F1 Team      Miami               10.0              HARD   
4       10033   Kick Sauber      Miami               10.0              HARD   

   pit_stops target_compound  time_loss_target  air_temp_mean  \
0          1          MEDIUM            37.060      26.573154   
1          2          MEDIUM            75.919      26.573154   
2          2          MEDIUM            75.287      26.573154   
3          1          MEDIUM            38.162      26.573154   
4          2          MEDIUM            74.296      26.573154   

   track_temp_mean  humidity_mean  rain_minutes_ratio  wind_speed_mean  
0        38.690604    

In [6]:
print("\nPreprocessing...")
 
# Label encoders with unknown handling
team_le = LabelEncoder()
track_le = LabelEncoder()
start_le = LabelEncoder()
tire_le = LabelEncoder()
stops_le = LabelEncoder()
 
team_le.fit(df["team_name"].astype(str))
track_le.fit(df["track_name"].astype(str))
start_le.fit(df["starting_compound"].astype(str))
tire_le.fit(df["target_compound"].astype(str))
stops_le.fit(df["pit_stops"].astype(str))
 
X_team = team_le.transform(df["team_name"].astype(str))
X_track = track_le.transform(df["track_name"].astype(str))
X_start_comp = start_le.transform(df["starting_compound"].astype(str))
 
# Numerical features with improved normalization
NUM_COLS = ["starting_position", "air_temp_mean", "track_temp_mean", "humidity_mean", "rain_minutes_ratio", "wind_speed_mean"]
X_num = df[NUM_COLS].values.astype("float32")
 
# Robust standardization (clip outliers before normalizing)
for i in range(X_num.shape[1]):
    col = X_num[:, i]
    q1, q99 = np.percentile(col, [1, 99])
    col = np.clip(col, q1, q99)
    mean, std = col.mean(), col.std()
    if std > 0:
        X_num[:, i] = (col - mean) / std
    else:
        X_num[:, i] = 0.0
 
# Weather sequences - pad to consistent length
max_timesteps = max(s.shape[0] for s in X_seq_raw)
X_seq = pad_sequences(X_seq_raw, maxlen=max_timesteps, padding="post", dtype="float32")
 
# Normalize weather sequences
seq_mean = X_seq.mean(axis=(0, 1))
seq_std = X_seq.std(axis=(0, 1)) + 1e-8
X_seq = (X_seq - seq_mean) / seq_std
 
# Targets
y_stops = stops_le.transform(df["pit_stops"].astype(str))
y_tire = tire_le.transform(df["target_compound"].astype(str))
y_time = df["time_loss_target"].values.astype("float32")
 
# Better time normalization (log transform for skewed distribution)
y_time_log = np.log1p(y_time)
y_time_mean = y_time_log.mean()
y_time_std = y_time_log.std() + 1e-8
y_time_norm = (y_time_log - y_time_mean) / y_time_std
 
# Train/val split with stratification
indices = np.arange(len(df))
train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=42, 
                                       stratify=y_stops)
 
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")



Preprocessing...


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

## Build and train GRU multi-task model

In [ ]:
print("\nBuilding improved model...")
 
# Increased capacity and regularization
EMBED_DIM = 16  # Increased from 8
GRU_UNITS = 64  # Increased from 32
DENSE_UNITS = 64  # Increased from 32
DROPOUT_RATE = 0.3  # Added dropout
L2_REG = 0.001  # Added L2 regularization
 
# Inputs
weather_seq_in = layers.Input(shape=(max_timesteps, len(WEATHER_FEATURES)), name="weather_seq")
team_in = layers.Input(shape=(1,), dtype="int32", name="team_in")
track_in = layers.Input(shape=(1,), dtype="int32", name="track_in")
start_comp_in = layers.Input(shape=(1,), dtype="int32", name="start_comp_in")
num_in = layers.Input(shape=(len(NUM_COLS),), name="num_in")
 
# Embeddings with regularization
team_emb = layers.Embedding(len(team_le.classes_), EMBED_DIM, 
                            embeddings_regularizer=regularizers.l2(L2_REG),
                            name="team_emb")(team_in)
team_emb = layers.Flatten()(team_emb)
 
track_emb = layers.Embedding(len(track_le.classes_), EMBED_DIM,
                             embeddings_regularizer=regularizers.l2(L2_REG),
                             name="track_emb")(track_in)
track_emb = layers.Flatten()(track_emb)
 
start_emb = layers.Embedding(len(start_le.classes_), EMBED_DIM,
                             embeddings_regularizer=regularizers.l2(L2_REG),
                             name="start_comp_emb")(start_comp_in)
start_emb = layers.Flatten()(start_emb)
 
# Bidirectional GRU with dropout
gru_out = layers.Bidirectional(
    layers.GRU(GRU_UNITS, return_sequences=False, 
               dropout=DROPOUT_RATE,
               recurrent_dropout=0.2,
               kernel_regularizer=regularizers.l2(L2_REG)),
    name="bi_gru"
)(weather_seq_in)
 
# Concatenate all features
concat = layers.Concatenate(name="concat_all")([gru_out, team_emb, track_emb, start_emb, num_in])
 
# Shared dense layers with dropout
shared = layers.Dense(DENSE_UNITS, activation="relu", 
                     kernel_regularizer=regularizers.l2(L2_REG),
                     name="shared_dense_1")(concat)
shared = layers.Dropout(DROPOUT_RATE)(shared)
shared = layers.Dense(DENSE_UNITS // 2, activation="relu",
                     kernel_regularizer=regularizers.l2(L2_REG),
                     name="shared_dense_2")(shared)
shared = layers.Dropout(DROPOUT_RATE)(shared)
 
# Task-specific heads
stops_out = layers.Dense(len(stops_le.classes_), activation="softmax", name="stops_out")(shared)
tire_out = layers.Dense(len(tire_le.classes_), activation="softmax", name="tire_out")(shared)
time_out = layers.Dense(1, activation="linear", name="time_out")(shared)
 
model = keras.Model(
    inputs={
        "weather_seq": weather_seq_in,
        "team_in": team_in,
        "track_in": track_in,
        "start_comp_in": start_comp_in,
        "num_in": num_in,
    },
    outputs={"stops_out": stops_out, "tire_out": tire_out, "time_out": time_out},
)
 
# Improved optimizer with gradient clipping and lower learning rate
optimizer = keras.optimizers.Adam(
    learning_rate=0.0005,  # Reduced from 0.002
    clipnorm=1.0  # Gradient clipping for stability
)
 
# Balanced multi-task loss weighting
model.compile(
    optimizer=optimizer,
    loss={
        "stops_out": "sparse_categorical_crossentropy",
        "tire_out": "sparse_categorical_crossentropy",
        "time_out": "mse",
    },
    loss_weights={
        "stops_out": 1.5,  # Slightly prioritize strategy
        "tire_out": 1.0,
        "time_out": 0.8,   # Reduce time loss weight (noisier target)
    },
    metrics={
        "stops_out": "accuracy",
        "tire_out": "accuracy",
        "time_out": "mae",
    },
)
 
model.summary()


In [ ]:
print("\nTraining...")
 
# Prepare training data
X_train = {
    "weather_seq": X_seq[train_idx],
    "team_in": X_team[train_idx],
    "track_in": X_track[train_idx],
    "start_comp_in": X_start_comp[train_idx],
    "num_in": X_num[train_idx],
}
 
y_train = {
    "stops_out": y_stops[train_idx],
    "tire_out": y_tire[train_idx],
    "time_out": y_time_norm[train_idx],
}
 
X_val = {
    "weather_seq": X_seq[val_idx],
    "team_in": X_team[val_idx],
    "track_in": X_track[val_idx],
    "start_comp_in": X_start_comp[val_idx],
    "num_in": X_num[val_idx],
}
 
y_val = {
    "stops_out": y_stops[val_idx],
    "tire_out": y_tire[val_idx],
    "time_out": y_time_norm[val_idx],
}
 
# Improved callbacks
callbacks = [
    # Early stopping with patience
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate on plateau
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    # Save best model
    ModelCheckpoint(
        MODEL_FILE,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
]
 
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=150,  # More epochs with early stopping
    batch_size=32,  # Increased from 16 for stability
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
print("\n============================================================")
print("TRAINING COMPLETE")
print("============================================================\n")

# Plot training history
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig("training_history_improved.png")
plt.show()

print("Saved training history plot: training_history_improved.png\n")

print("============================================================")
print("FINAL VALIDATION METRICS")
print("============================================================")
final_loss = history.history['val_loss'][-1]
final_stops_acc = history.history['val_stops_out_accuracy'][-1]
final_tire_acc = history.history['val_tire_out_accuracy'][-1]
final_time_mae = history.history['val_time_out_mae'][-1]
print(f"Total Loss: {final_loss:.4f}")
print(f"Stops Accuracy: {final_stops_acc * 100:.2f}%")
print(f"Tire Accuracy: {final_tire_acc * 100:.2f}%")
print(f"Time MAE (normalized): {final_time_mae:.4f}")
